In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: e:\Gen Ai Course\Projects\ML Project\LLM-Powered RAG Document Assistant


In [2]:
%pip install langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from src.llm import get_llm

print("LLM module imported successfully.")

LLM module imported successfully.


In [6]:
llm = get_llm(
    model_name="openai/gpt-oss-20b",
    temperature=0.0,
)

print("LLM initialized successfully.")

LLM initialized successfully.


In [7]:
response = llm.invoke(
    "Explain what Retrieval-Augmented Generation is in one sentence."
)

print(response.content)

Retrieval‑Augmented Generation is a method that blends a language model with an external search engine, pulling in relevant documents at inference time so the model can generate answers grounded in up‑to‑date, specific knowledge.


In [8]:
assert response.content
assert isinstance(response.content, str)

print("LLM test passed.")

LLM test passed.


In [9]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: e:\Gen Ai Course\Projects\ML Project\LLM-Powered RAG Document Assistant


In [10]:
from src.embeddings import get_embedding_model
from src.vector_store import load_vector_store
from src.retriever import create_retriever
from src.llm import get_llm
from src.rag_pipeline import generate_rag_response

from config.settings import TOP_K

In [11]:
embedding_model = get_embedding_model()

vector_store = load_vector_store(
    embedding_model=embedding_model
)

retriever = create_retriever(
    vector_store=vector_store,
    top_k=TOP_K,
)

llm = get_llm(
    model_name="openai/gpt-oss-20b",
    temperature=0.0,
)

print("RAG components loaded successfully.")

e:\Gen Ai Course\Projects\ML Project\LLM-Powered RAG Document Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8549.13it/s]


RAG components loaded successfully.


In [12]:
question = (
    "What are some potential environmental impacts "
    "associated with oil and gas development?"
)

result = generate_rag_response(
    llm=llm,
    retriever=retriever,
    question=question,
)

print(result["answer"])

Potential environmental impacts associated with oil and gas development include:

| Impact | Description | Source |
|--------|-------------|--------|
| **Greenhouse‑gas (GHG) emissions** | Upstream production and downstream use of crude oil can generate significant GHG emissions. The EIS estimates that the emissions from the oil exported at maximum capacity are large, though most of them already occur as part of the U.S. economy. | Source 3 (Page 58) |
| **Downstream emissions from refining and combustion** | The project assumes all exported crude will be refined into gasoline and diesel and then combusted in passenger vehicles, which would produce additional emissions. | Source 1 (Page 797) |
| **Potential loss to fishing and related industries** | Construction and operation of the project could negatively affect fishing or other related industries, though such losses are possible rather than guaranteed. | Source 2 (Page 808) |
| **Limited long‑term productivity or environmental gains

In [13]:
print("Sources used:")
print()

for i, document in enumerate(
    result["source_documents"],
    start=1,
):
    print(
        f"{i}. "
        f"{document.metadata.get('source_file', 'Unknown')} "
        f"| Page: "
        f"{document.metadata.get('page', 'Unknown')}"
    )

Sources used:

1. eis_Sea_Port_Oil_Terminal.pdf | Page: 797
2. eis_Sea_Port_Oil_Terminal.pdf | Page: 808
3. eis_Sea_Port_Oil_Terminal.pdf | Page: 58
4. eis_Fort_Wainwright_Alaska.pdf | Page: 289


In [14]:
assert result["answer"]
assert isinstance(result["answer"], str)

assert result["source_documents"]
assert len(result["source_documents"]) == TOP_K

print("RAG pipeline validation passed.")

RAG pipeline validation passed.


In [15]:
question_unknown = (
    "What is the population of Mars according to the "
    "provided environmental documents?"
)

result_unknown = generate_rag_response(
    llm=llm,
    retriever=retriever,
    question=question_unknown,
)

print(result_unknown["answer"])

I could not find the answer in the provided documents.


In [16]:
question_known = (
    "What are some potential environmental impacts "
    "associated with oil and gas development?"
)

result_known = generate_rag_response(
    llm=llm,
    retriever=retriever,
    question=question_known,
)

print(result_known["answer"])

Potential environmental impacts associated with oil and gas development include:

| Impact | Description | Source |
|--------|-------------|--------|
| **Greenhouse‑gas (GHG) emissions** | Upstream production and downstream use of crude oil can generate significant GHG emissions. The EIS estimates that the emissions from the oil exported at maximum capacity are large, though most of them already occur as part of the U.S. economy. | Source 3 (Page 58) |
| **Downstream emissions from refining and combustion** | The project assumes all exported crude will be refined into gasoline and diesel and then combusted in passenger vehicles, which would produce additional emissions. | Source 1 (Page 797) |
| **Potential loss to fishing and related industries** | Construction and operation of the project could negatively affect fishing or other related industries, though such losses are possible rather than guaranteed. | Source 2 (Page 808) |
| **Limited long‑term productivity or environmental gains

In [17]:
print("\nSources:")
for i, document in enumerate(
    result_known["source_documents"],
    start=1,
):
    print(
        f"{i}. "
        f"{document.metadata.get('source_file', 'Unknown')} "
        f"| Page: "
        f"{document.metadata.get('page', 'Unknown')}"
    )


Sources:
1. eis_Sea_Port_Oil_Terminal.pdf | Page: 797
2. eis_Sea_Port_Oil_Terminal.pdf | Page: 808
3. eis_Sea_Port_Oil_Terminal.pdf | Page: 58
4. eis_Fort_Wainwright_Alaska.pdf | Page: 289
